# Confronto BioMamba vs Corponi (WESAD)

## Obiettivo
Questo notebook esegue il fine-tuning supervisionato del modello **BioMamba** sul dataset **WESAD** partendo dai pesi pre-addestrati nella Phase 1.

In [ ]:
# 1. Setup Ambiente (Kaggle-Final-Fix)
import os, sys, shutil
!pip install -q --upgrade pip
!pip install -q packaging setuptools triton

print("Installazione causal-conv1d...")
!pip install -q causal-conv1d>=1.4.0 --no-build-isolation

print("Installazione mamba-ssm...")
%env MAX_JOBS=1
!pip install -q mamba-ssm --no-build-isolation

print("Installazione dipendenze generali...")
!pip install -q einops biosppy

import os
os._exit(0)

In [ ]:
# 2. Caricamento Pesi e Architettura
import os, glob, pickle, torch, torch.nn as nn, numpy as np, pandas as pd
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from einops import rearrange
from sklearn.metrics import accuracy_score, f1_score, classification_report
from mamba_ssm import Mamba

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Cerchiamo i pesi pre-addestrati
weights_files = glob.glob('/kaggle/input/**/biomamba_encoder_pretrained.pt', recursive=True)
WEIGHTS_PATH = weights_files[0] if weights_files else None

class BioMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.mamba_fwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_bwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        fwd_out = self.mamba_fwd(x)
        bwd_out = self.mamba_bwd(x.flip(dims=[1])).flip(dims=[1])
        return self.norm(fwd_out + bwd_out) + x

class BioMambaEncoder(nn.Module):
    def __init__(self, n_channels=6, d_model=128, n_layers=4):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Linear(n_channels, d_model)
        self.layers = nn.ModuleList([BioMambaBlock(d_model) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        x = rearrange(x, 'b c t -> b t c')
        x = self.embedding(x)
        for layer in self.layers: x = layer(x)
        return self.norm(x).mean(dim=1)

class BioMambaClassifier(nn.Module):
    def __init__(self, encoder, n_classes=2, dropout=0.5):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(encoder.d_model, encoder.d_model // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(encoder.d_model // 2, n_classes))
    def forward(self, x): return self.head(self.encoder(x))

encoder = BioMambaEncoder()
if WEIGHTS_PATH:
    print(f"✅ Caricamento pesi pre-addestrati da: {WEIGHTS_PATH}")
    encoder.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device), strict=False)
else:
    print("⚠️ ATTENZIONE: Pesi pre-addestrati non trovati.")

model = BioMambaClassifier(encoder).to(device)


In [ ]:
# 3. Caricamento Dati WESAD
WESAD_DIR = glob.glob('/kaggle/input/**/WESAD', recursive=True)[0]
TARGET_LEN, LABEL_MAP = 640, {1: 0, 2: 1}

def segment_subject(data, label_signal):
    wrist = data['signal']['wrist']
    bvp, eda, temp, acc = wrist['BVP'].flatten(), wrist['EDA'].flatten(), wrist['TEMP'].flatten(), wrist['ACC']
    lbl = resample(label_signal.astype(float), len(bvp)).round().astype(int)
    segs, labs = [], []
    for start in range(0, len(bvp) - TARGET_LEN + 1, TARGET_LEN):
        maj = int(np.bincount(lbl[start:start+TARGET_LEN][np.isin(lbl[start:start+TARGET_LEN], [1,2])]).argmax()) if len(lbl[start:start+TARGET_LEN][np.isin(lbl[start:start+TARGET_LEN], [1,2])]) > 0 else 0
        if maj not in LABEL_MAP: continue
        arr = np.stack([resample(acc[start//2:(start+TARGET_LEN)//2, i], TARGET_LEN) for i in range(3)] + [bvp[start:start+TARGET_LEN], resample(eda[start//16:(start+TARGET_LEN)//16], TARGET_LEN), resample(temp[start//16:(start+TARGET_LEN)//16], TARGET_LEN)])
        arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-8)
        segs.append(arr); labs.append(LABEL_MAP[maj])
    return segs, labs

all_X, all_y, all_subs = [], [], []
for sdir in sorted(glob.glob(os.path.join(WESAD_DIR, 'S*'))):
    sid = os.path.basename(sdir)
    with open(os.path.join(sdir, f'{sid}.pkl'), 'rb') as f: data = pickle.load(f, encoding='latin1')
    segs, labs = segment_subject(data, data['label'])
    all_X.extend(segs); all_y.extend(labs); all_subs.extend([sid]*len(segs))


In [ ]:
# 4. Training Loop e Risultati
u_subs = np.unique(all_subs); np.random.seed(42); np.random.shuffle(u_subs)
tr_s, te_s = u_subs[:int(0.7*len(u_subs))], u_subs[int(0.7*len(u_subs)):]
tr_idx = np.isin(all_subs, tr_s)

class SigDataset(Dataset):
    def __init__(self, m): self.X, self.y = torch.tensor(np.array(all_X)[m]), torch.tensor(np.array(all_y)[m])
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

weights = 1. / np.array([len(np.where(np.array(all_y)[tr_idx] == t)[0]) for t in np.unique(np.array(all_y)[tr_idx])])
sampler = WeightedRandomSampler(torch.tensor([weights[int(t)] for t in np.array(all_y)[tr_idx]]), len(np.where(tr_idx)[0]))
train_loader = DataLoader(SigDataset(tr_idx), batch_size=32, sampler=sampler)
test_loader = DataLoader(SigDataset(np.isin(all_subs, te_s)), batch_size=32)

opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01); crit = nn.CrossEntropyLoss()
print("Starting Fine-tuning (50 epochs)...")
for e in range(1, 51):
    model.train()
    for xb, yb in train_loader: opt.zero_grad(); crit(model(xb.float().to(device)), yb.to(device)).backward(); opt.step()
    if e % 10 == 0: print(f"Epoch {e}/50 completata.")

model.eval(); p, t = [], []
with torch.no_grad():
    for xb, yb in test_loader: p.extend(model(xb.float().to(device)).argmax(1).cpu().numpy()); t.extend(yb.numpy())

acc = accuracy_score(t, p)
f1 = f1_score(t, p)
f1_macro = f1_score(t, p, average='macro')
print('\n--- CONFRONTO FINALE BioMamba vs Corponi (WESAD) ---')
df = pd.DataFrame({
    'Modello': ['Corponi (Master)', 'BioMamba (2025)'],
    'Accuracy': ['80.20%', f'{acc*100:.2f}%'],
    'F1-Score (Stress)': ['54.55%', f'{f1*100:.2f}%'],
    'Macro F1': ['70.95%', f'{f1_macro*100:.2f}%']
})
print(df.to_string(index=False))
